# Needle 2 日本語 LoRA ファインチューニング

b.ai API で日本語のツール呼び出しデータを生成し、Google Colab 上で Needle 2 を LoRA 学習します。学習データ、LoRA アダプター、メタデータ、`.cact` を Google Drive に保存します。

> 事前準備: Colab の「ランタイム」→「ランタイムのタイプを変更」から GPU（T4 など）を選択してください。左側の Secrets に `BAI_API_KEY` を登録してください。

## 1. Needle と依存関係を準備

In [ ]:
import os

%cd /content
if not os.path.exists('/content/needle'):
    !git clone https://github.com/msArray/needle
%cd /content/needle
!pip install -e ".[test]" -q
!python -c "import needle; print('Needle import: OK')"

## 2. b.ai API と学習設定

API キーはセルに直接書かず、Colab Secrets の `BAI_API_KEY` から読み込みます。`BAI_MODEL` は b.ai で利用できるモデル名に変更してください。

In [ ]:
import os
from google.colab import userdata

os.environ['BAI_API_KEY'] = userdata.get('BAI_API_KEY')
os.environ['BAI_API_URL'] = 'https://api.b.ai/v1/chat/completions'
os.environ['BAI_MODEL'] = 'your-model-name'  # b.ai の利用可能なモデル名に変更

# まず小規模に確認し、本番では NUM_SAMPLES / EPOCHS を増やします。
NUM_SAMPLES = 100
EPOCHS = 3
BATCH_SIZE = 4
MAX_LEN = 512
LORA_RANK = 16
LORA_ALPHA = 32
WORKERS = 4
LANGUAGE = 'ja'
print('b.ai endpoint:', os.environ['BAI_API_URL'])
print('model:', os.environ['BAI_MODEL'])

## 3. `tools.json` をアップロード

実際に Needle から呼び出すツールの JSON スキーマをアップロードしてください。

In [ ]:
from google.colab import files

uploaded = files.upload()
tool_files = [name for name in uploaded if name.endswith('.json')]
if not tool_files:
    raise ValueError('tools.json などの JSON ファイルをアップロードしてください')
TOOLS_PATH = '/content/needle/tools.json'
os.replace('/content/needle/' + tool_files[0], TOOLS_PATH) if os.path.exists('/content/needle/' + tool_files[0]) else os.replace('/content/' + tool_files[0], TOOLS_PATH)
print('tools:', TOOLS_PATH)

## 4. 日本語データを生成

最初は `NUM_SAMPLES=100` 程度で出力を確認してください。生成データは学習前に必ず検査します。

In [ ]:
import subprocess

DATA_PATH = '/content/needle/data_ja.jsonl'
cmd = [
    'needle', 'generate-data', '--tools', TOOLS_PATH,
    '--num-samples', str(NUM_SAMPLES), '--batch-size', '10',
    '--workers', str(WORKERS), '--language', LANGUAGE,
    '--output', DATA_PATH,
]
subprocess.run(cmd, check=True)
print('generated:', DATA_PATH)
!head -n 3 /content/needle/data_ja.jsonl

## 5. データの簡易検査

In [ ]:
import json

rows = []
with open(DATA_PATH, encoding='utf-8') as f:
    for line_number, line in enumerate(f, 1):
        row = json.loads(line)
        assert row.get('query'), f'query が空です: {line_number} 行目'
        assert 'answers' in row, f'answers がありません: {line_number} 行目'
        rows.append(row)
print('valid examples:', len(rows))
print('off-topic examples:', sum(not row['answers'] for row in rows))
print('query sample:', rows[0]['query'])
# 内容が不正な場合はここで停止し、data_ja.jsonl を修正してから次へ進みます。

## 6. LoRA 学習

Colab の GPU メモリが少ない場合は `BATCH_SIZE` や `MAX_LEN` を下げてください。

In [ ]:
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
ADAPTER_PATH = '/content/needle/checkpoints/needle_ja_lora.pkl'
cmd = [
    'needle', 'finetune', DATA_PATH, '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE), '--lora-rank', str(LORA_RANK),
    '--lora-alpha', str(LORA_ALPHA), '--max-len', str(MAX_LEN),
    '--val-split', '0.1', '--language', LANGUAGE, '--out', ADAPTER_PATH,
]
subprocess.run(cmd, check=True)
print('adapter:', ADAPTER_PATH)
print('dataset copy:', ADAPTER_PATH + '.dataset.jsonl')
print('metadata:', ADAPTER_PATH + '.metadata.json')

## 7. `.cact` に書き出し

In [ ]:
CACT_PATH = '/content/needle/needle_ja.cact'
subprocess.run([
    'needle', 'build', 'checkpoints/needle2.pkl', '--lora', ADAPTER_PATH,
    '--out', CACT_PATH, '--bits', '2',
], check=True)
print('cact:', CACT_PATH)

## 8. Google Drive に保存

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')
RUN_DIR = '/content/drive/MyDrive/needle_runs/japanese_needle_run'
os.makedirs(RUN_DIR, exist_ok=True)
artifacts = [
    TOOLS_PATH, DATA_PATH, ADAPTER_PATH, ADAPTER_PATH + '.dataset.jsonl',
    ADAPTER_PATH + '.metadata.json', CACT_PATH,
]
for path in artifacts:
    if os.path.exists(path):
        shutil.copy2(path, os.path.join(RUN_DIR, os.path.basename(path)))
print('saved to:', RUN_DIR)
!ls -lh /content/drive/MyDrive/needle_runs/japanese_needle_run

## 9. ローカルへダウンロード（任意）

In [ ]:
from google.colab import files
files.download(CACT_PATH)
# 学習データやメタデータも必要なら、次を個別に実行します。
# files.download(DATA_PATH)
# files.download(ADAPTER_PATH + '.metadata.json')